# 单通道批量：出 trace 库

把一组 sif 逐个跑完 **读入 → 相机 → 裁切 → 定位 → 漂移 → emitter → trace**，
不做任何 trace 筛选（状态判定/挑选留给下游）。参数与 `smTest.ipynb` 一致，
读取与缓存的组织方式与 `dualCh/smTest_dual_batch.ipynb` 一致。

**这次的数据帧数可能不同**，所以帧数一律用文件本身的全长（`FRAME_STOP = None`），
不在这里统一截断。

每个视野写到 `<sif 同目录>/<sif 主名>_trace/`：

| 文件 | 内容 |
|---|---|
| `traces.npz` | **完整 trace 库**：`emitter_id / x_px / y_px / intensity(N,T) / bg_flux(N,T) / t_s(T)` + 标量元信息 |
| `emitters.csv` | 逐分子统计（`collect_statistics`：n_locs / photons / σ / 偏心率 …） |
| `traces_heatmap.png` | 全部 trace 一屏看完（按亮度排序的 intensity 热图 + 群体中位曲线） |
| `traces_pages/traces_pNNN.png` | 逐条曲线分页（每页 48 条） |

这一本**只跑批、只出 trace 库**；筛选 / 池化 / 统计另开 notebook 做（读各目录的 `traces.npz`）。

定位缓存写在 sif 旁边（`<主名>.locs.npz`，与 `smTest.ipynb` 同一处；`READ_LOC_CACHE=True` 时只读）。


In [1]:
# `import smlm` 需要 Py_Smile 在 sys.path 上; notebook 放在子目录里时自己找一下
import pathlib
import sys

for _q in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_q / 'smlm' / '__init__.py').is_file():
        sys.path.insert(0, str(_q))
        break

import gc
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import smlm.trace as tr
from smlm import viz
from smlm.camera import EMCCD
from smlm.datatypes import DriftTrace
from smlm.drift import CloudDriftEstimator
from smlm.emitter import analyze_emitters_sm, collect_statistics
from smlm.imgprocess import Crop, DriftCorrector, Pipeline, SelectFrames
from smlm.io import load_raw
from smlm.localize import localize_stack, psf_sigma

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ======================= 输入 =======================
SIF_FILES = [
    r"G:\SPARK\20261124\Rhod_avidin_532nm_2mW_0p1S_2000frames_fov1.sif",
    r"G:\SPARK\20261124\Rhod_avidin_532nm_2mW_0p1S_4000frames_fov2.sif",
    r"G:\SPARK\20261124\Rhod_avidin_532nm_4mW_0p1S_2000frames_fov1.sif",
    r"G:\SPARK\20261124\Rhod_avidin_532nm_6mW_0p1S_2000frames_fov1.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_6mW_0p1S_1000frames_fov1.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_6mW_0p1S_2000frames_fov2.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_6mW_0p1S_2000frames_fov3.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_6mW_0p1S_2000frames_fov4.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_8mW_0p1S_1000frames_fov1.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_8mW_0p1S_1000frames_fov2.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_8mW_0p1S_1000frames_fov3.sif",
    r"G:\SPARK\20261124\SPARK Cy3_avidin_532nm_8mW_0p1S_1000frames_fov4.sif",
]

SIF_FILES = [pathlib.Path(p) for p in SIF_FILES]
# 只算个别文件时填文件名子串, 例如 ['fov3']; 空 = 全部
ONLY_FILES = []

OUT_TAG = ''            # 想保住上一轮就改成 '_v1'

# ======================= 相机 / 裁切 (同 smTest) =======================
CAM_KW = dict(conv_factor=5.75, em_gain=300)
PIXEL_SIZE_NM = 106.6
CROP_Y, CROP_X = (0, 512), (0, 512)     # 超出图像时自动夹住
FRAME_STOP = None                       # None = 文件全长 (这批帧数不一致, 不截断)

# ======================= 定位 (同 smTest cell 12) =======================
READ_LOC_CACHE = False                  # False = 重算并覆盖缓存; True = 只读缓存(缺了报错)
K_SIGMA = 2.6
DET_KW = dict(edge=5, f_em=np.sqrt(2), bg_window=15,
              max_size=7, min_neighbors=2, verbose=False)
LOC_KW = dict(edge=5, F_EM=np.sqrt(2.0), use_pixel=False,
              use_gradient=True, n_jobs=12, verbose=False)
FILT_KW = dict(photons=(20, 1e7), bg=(0, 1e5),
               sigma=(0.5, 5.0), lp=(0.0, 1.0))

# ======================= 漂移 (同 smTest cell 21) =======================
USE_DRIFT = True
FRAMES_PER_SEG = 10
DO_STAGE2 = True
DRIFT_CROP_MARGIN = 200                 # 每边收 200//2 = 100 px 做漂移缓冲
DRIFT_EDGE = 5

# ======================= emitter (同 smTest cell 26) =======================
EMITTER_KW = dict(search_radius=0.6, max_gap=1000, min_frames=5, ecc_p=1e-3,
                  motion='static', do_merge=True)
FIRST_FRAME_RANGE = (0, 200)             # 首次出现帧窗口 (smTest 里就是 (0,50))

# ======================= trace (同 smTest cell 31) =======================
TRACE_K_SIGMA = 2.0                     # ApertureRaw 的 k_sigma (半径 = ceil(k·σ))
BG_GAP, BG_WIDTH, BG_EXCL = 2, 4, 2     # AnnulusRing
TRACE_PAGE = 48                         # 每页画多少条曲线
TRACE_PAGES_MAX = 20                    # 分页图最多写多少页 (库本身是全的)

# ======================= 校验 =======================
missing = [str(p) for p in SIF_FILES if not p.is_file()]
if missing:
    raise FileNotFoundError('缺少 sif:\n  ' + '\n  '.join(missing))
TODO = [p for p in SIF_FILES
        if not ONLY_FILES or any(k in p.name for k in ONLY_FILES)]
print(f'待处理 : {len(TODO)} / {len(SIF_FILES)} 个 sif')
print(f'裁切   : y{CROP_Y} x{CROP_X}   帧数: '
      f'{"全长" if FRAME_STOP is None else f"前 {FRAME_STOP} 帧"}')
print('汇总   : 不做 (下游另开 notebook 读 traces.npz)')


待处理 : 12 / 12 个 sif
裁切   : y(0, 512) x(0, 512)   帧数: 全长
汇总   : G:\SPARK\20261124\TRACE_summary


## 逐文件处理

In [2]:
_T0 = 0.0


def _p(msg):
    """带耗时戳的进度行 (相对当前文件开始)。"""
    print(f'  {time.time() - _T0:6.1f}s  {msg}', flush=True)


def _plot_grid(traces, t_s, path, *, page=48, ncols=8, dpi=150, title=''):
    """一页 = 若干条 intensity 曲线 (每页 ncols×ceil(page/ncols) 个小图)。"""
    idx = list(range(len(traces)))
    nrow = int(np.ceil(len(idx) / ncols))
    fig, axes = plt.subplots(nrow, ncols, figsize=(1.45 * ncols, 1.25 * nrow),
                             dpi=dpi, squeeze=False, sharex=True)
    for k, ax in enumerate(axes.ravel()):
        if k >= len(idx):
            ax.axis('off')
            continue
        t = traces[idx[k]]
        ax.plot(t_s, np.asarray(t['intensity'], float), lw=0.6, color='tab:blue')
        ax.axhline(0.0, color='0.85', lw=0.4)
        ax.set_title(f"#{t['emitter_id']}", fontsize=5.5, pad=1.5)
        ax.tick_params(labelsize=4.5, length=1.5, pad=1)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
    if title:
        fig.suptitle(title, fontsize=8, y=1.0)
    fig.tight_layout(pad=0.3)
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def _plot_heatmap(traces, t_s, path, *, dpi=150, title=''):
    """全部 trace 一屏: 按峰值亮度排序的 intensity 热图 + 群体中位曲线。"""
    I = np.vstack([np.asarray(t['intensity'], float) for t in traces])
    order = np.argsort(-np.nanpercentile(I, 95, axis=1))
    I = I[order]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), dpi=dpi,
                             gridspec_kw=dict(width_ratios=[3, 1]))
    im = axes[0].imshow(I, aspect='auto', cmap='magma',
                        extent=[t_s[0], t_s[-1], len(I), 0],
                        vmin=0.0, vmax=float(np.nanpercentile(I, 99)))
    axes[0].set_xlabel('time (s)', fontsize=8)
    axes[0].set_ylabel(f'trace (sorted by brightness, n={len(I)})', fontsize=8)
    axes[0].tick_params(labelsize=7)
    fig.colorbar(im, ax=axes[0], pad=0.01, label='photons / frame')
    axes[1].plot(t_s, np.nanmedian(I, axis=0), color='0.2', lw=0.8)
    axes[1].set_title('population median', fontsize=8)
    axes[1].set_xlabel('time (s)', fontsize=8)
    axes[1].tick_params(labelsize=7)
    axes[1].grid(alpha=0.25)
    if title:
        fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def process_one(src, *, verbose=True):
    """跑一个 sif, 成品写进 <sif 同目录>/<主名>_trace{OUT_TAG}/。返回汇总表的一行。"""
    global _T0
    _T0 = time.time()
    src = pathlib.Path(src)
    out_dir = src.with_suffix('').with_name(src.stem + '_trace' + OUT_TAG)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- 1) sif -> 光子 -> 裁切 (帧数用全长) ----
    raw = load_raw(src)
    raw.metadata.pop('imageData', None)
    ex_time = float(raw.exposure_time)
    image = EMCCD(**CAM_KW).process(raw, pixel_size=PIXEL_SIZE_NM)
    del raw
    h, w = image.shape[-2:]
    crop_y = (CROP_Y[0], min(CROP_Y[1], h))
    crop_x = (CROP_X[0], min(CROP_X[1], w))
    if crop_y != tuple(CROP_Y) or crop_x != tuple(CROP_X):
        _p(f'[!] 图像只有 {h}x{w}, 裁切被夹到 y{crop_y} x{crop_x}')
    stack = Pipeline(Crop(y=crop_y, x=crop_x),
                     SelectFrames(start=0, stop=FRAME_STOP))(image)
    del image
    _p(f'读入 : {stack.n_frames} 帧  {ex_time * 1000:.0f} ms/帧  {stack.shape[1:]}')

    # ---- 2) 定位 (带缓存: 与 smTest 一致, 缓存就写在 sif 旁边) ----
    locs = localize_stack(stack, k_sigma=K_SIGMA, det_kw=DET_KW, loc_kw=LOC_KW,
                          filt_kw=FILT_KW, src_path=src,
                          read_cache=READ_LOC_CACHE, verbose=False, tag='sc')
    _p(f'定位 : {len(locs)} locs   σ 中位 '
       f'{float(np.median(locs.sigma)):.2f} px')

    # ---- 3) 漂移 ----
    locs_drift = locs[locs.frame < stack.n_frames]
    if USE_DRIFT:
        uncertainty = float(np.median(locs_drift.lp))
        est = CloudDriftEstimator(frames_per_seg=FRAMES_PER_SEG,
                                  uncertainty=uncertainty, do_stage2=DO_STAGE2)
        drift = est.estimate(locs_drift, n_frames=stack.n_frames)
    else:
        drift = DriftTrace.zeros(stack.n_frames, pixel_size=PIXEL_SIZE_NM)
    hs, ws = int(stack.shape[1]), int(stack.shape[2])
    crop = (max(8, hs - DRIFT_CROP_MARGIN), max(8, ws - DRIFT_CROP_MARGIN))
    corr_image, corr_locs, _ = DriftCorrector(crop=crop, edge=DRIFT_EDGE).apply(
        trace=drift, image=stack, locs=locs_drift)
    n_locs = len(corr_locs)
    corr_h, corr_w = (int(v) for v in corr_image.shape[1:])
    del stack, locs, locs_drift
    _p(f'漂移 : 累计 {drift.total_drift:.2f} px -> {corr_h}x{corr_w}  '
       f'locs {n_locs}')

    # ---- 4) emitter ----
    n_frames = int(corr_image.n_frames)
    emitters, _ = analyze_emitters_sm(
        corr_locs, n_frames=n_frames, first_frame_range=FIRST_FRAME_RANGE,
        **EMITTER_KW)
    stats = collect_statistics(emitters, corr_locs, ex_time, 0)
    _p(f'emitter: {len(emitters)} 个 (from {len(corr_locs)} locs)')

    # ---- 5) trace (σ 用拟合宽度中位, 与双通道同一口径) ----
    sigma_px = psf_sigma(corr_locs)
    centers_yx = np.array([[e.centroid[1], e.centroid[0]] for e in emitters])
    pipe = tr.TraceFillPipeline(corr_image.data, centers_yx, sigma=sigma_px,
                                n_frames=n_frames, tag='sc')
    traces = pipe.run([tr.ApertureRaw(k_sigma=TRACE_K_SIGMA),
                       tr.AnnulusRing(gap=BG_GAP, width=BG_WIDTH,
                                      excl_radius=BG_EXCL),
                       tr.IntensityFromRawBg(),
                       tr.PoissonSNR()], verbose=False)
    del corr_image, corr_locs
    _p(f'trace : {len(traces)} 条 x {n_frames} 帧  '
       f'(σ={sigma_px:.2f} px, 半径 {traces[0]["radius"]} px)')

    # ---- 6) 落盘: trace 库 + 逐分子统计 + 图 ----
    t_s = np.arange(n_frames) * ex_time
    np.savez_compressed(
        out_dir / 'traces.npz',
        emitter_id=np.array([int(t['emitter_id']) for t in traces], int),
        x_px=np.array([float(t['x']) for t in traces], float),
        y_px=np.array([float(t['y']) for t in traces], float),
        intensity=np.vstack([np.asarray(t['intensity'], np.float32) for t in traces]),
        bg_flux=np.vstack([np.asarray(t['bg_flux'], np.float32) for t in traces]),
        t_s=t_s,
        sigma_px=sigma_px, exposure_s=ex_time, n_frames=n_frames,
        source=src.name)
    pd.DataFrame({
        'idx': np.arange(len(emitters)),
        'x_px': np.array([e.centroid[0] for e in emitters]),
        'y_px': np.array([e.centroid[1] for e in emitters]),
        'n_locs': np.asarray(stats.n_locs),
        'survival_s': np.asarray(stats.survival_sec),
        'duty_cycle': np.asarray(stats.duty_cycle),
        'eccentricity': np.asarray(stats.eccentricity),
        'photons_sum': np.asarray(stats.photons_sum),
        'photons_per_s': np.asarray(stats.photons_per_sec),
        'sigma_mean': np.asarray(stats.sigma_mean),
        'bg_mean': np.asarray(stats.bg_mean),
    }).to_csv(out_dir / 'emitters.csv', index=False, encoding='utf-8-sig')

    if traces:
        _plot_heatmap(traces, t_s, out_dir / 'traces_heatmap.png',
                      title=f'{src.stem}   n={len(traces)}')
        pages = out_dir / 'traces_pages'
        pages.mkdir(exist_ok=True)
        n_page = int(np.ceil(len(traces) / TRACE_PAGE))
        for k in range(min(n_page, int(TRACE_PAGES_MAX))):
            chunk = traces[k * TRACE_PAGE:(k + 1) * TRACE_PAGE]
            _plot_grid(chunk, t_s, pages / f'traces_p{k + 1:03d}.png',
                       title=f'{src.stem}   page {k + 1}/{n_page}')
        if n_page > TRACE_PAGES_MAX:
            _p(f'[!] 分页图只写了前 {TRACE_PAGES_MAX}/{n_page} 页 (库是全的)')
    _p(f'完成 : {out_dir.name}/  ({len(traces)} 条 trace)')

    return dict(file=src.name, n_frames=n_frames, height=corr_h, width=corr_w,
                n_locs=n_locs, n_emitters=len(emitters),
                sigma_px=float(sigma_px), drift_px=float(drift.total_drift),
                seconds=round(time.time() - _T0, 1),
                out_dir=str(out_dir), error='')


## 跑整批

这一段只负责"把每个视野跑完、产物写到各自目录"。**不做汇总、不筛 trace** ——
交接物就是每个视野的 `traces.npz`，筛选 / 池化 / 统计另开 notebook 做。


In [ ]:
done, failed = [], []

for k, src in enumerate(TODO, 1):
    print(f'\n=== [{k}/{len(TODO)}] {src.name} ' + '=' * max(0, 36 - len(src.name)))
    try:
        row = process_one(src)
        done.append((row['file'], row['n_emitters'], row['n_frames']))
    except Exception as exc:                     # 单个文件失败不拖垮整批
        traceback.print_exc()
        failed.append(f'{src.name}: {type(exc).__name__}: {exc}')
    gc.collect()

print(f'\n完成 {len(done)} / {len(TODO)}')
if failed:
    print('失败:')
    for f in failed:
        print('  -', f)
